In [ ]:
!pip install penaltyblog

  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.8/63.8 kB 2.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.9/5.9 MB 64.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.1/56.1 kB 4.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.6/57.6 kB 4.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 58.0/58.0 kB 4.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 70.8/70.8 kB 5.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 74.8/74.8 kB 6.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.9/4.9 MB 103.0 MB/s eta 0:00:00
  Created wheel for socks: filename=socks-0-py3-none-any.whl size=960 sha256=745f589b4bef413b9abb55c21a79d6e2b2724c242212f206a0e9f67d8b1ddb37
  Stored in directory: /root/.cache/pip/wheels/23/ce/b7/b686a9d7fece21d2f1ea52af48af15f9aae43165beb946ce2a
Successfully built socks


In [1]:
"""
Project B12: 2019-2026 Data & Baseline Metric Extraction Script
Run this script in your Colab/Jupyter environment to generate exact numerical values
for your presentation slides, report, and defense dashboard.
"""

import pandas as pd
import numpy as np
import penaltyblog as pb

def main():
    print("=" * 60)
    print("PROJECT B12: EXTRACTING 2019-2026 DATASET METRICS")
    print("=" * 60)

    leagues = ["ENG Premier League", "ESP La Liga", "DEU Bundesliga 1",
               "ITA Serie A", "FRA Ligue 1", "NLD Eredivisie", "PRT Liga 1", "TUR Super Lig",
    ]
    seasons = [
        "2019-2020", "2020-2021", "2021-2022",
        "2022-2023", "2023-2024", "2024-2025", "2025-2026"
    ]

    dataframes = []
    print("\n[1/4] Scraping/Ingesting match data across 7 seasons...")
    for league in leagues:
        for season in seasons:
            try:
                fb = pb.scrapers.FootballData(league, season)
                df_temp = fb.get_fixtures()
                dataframes.append(df_temp)
                print(f"  ✓ Ingested {league} {season}")
            except Exception as e:
                print(f"  ✗ Could not ingest {league} {season}: {e}")

    # Concatenate all dataframes efficiently (O(N) memory allocation)
    df = pd.concat(dataframes, ignore_index=True)

    # Preprocessing & Data Sanitization
    print("\n[2/4] Sanitizing and chronologically sorting dataset...")
    df = df.dropna(subset=['goals_home', 'goals_away', 'datetime'])
    df['date'] = pd.to_datetime(df['datetime'])
    df = df.sort_values('date').reset_index(drop=True)

    total_rows = len(df)
    null_count = df[['goals_home', 'goals_away', 'team_home', 'team_away']].isnull().sum().sum()

    home_goals_avg = df['goals_home'].mean()
    away_goals_avg = df['goals_away'].mean()

    home_wins = len(df[df['goals_home'] > df['goals_away']])
    away_wins = len(df[df['goals_home'] < df['goals_away']])
    draws = len(df[df['goals_home'] == df['goals_away']])

    print("\n" + "=" * 60)
    print("EMPIRICAL DATASET SUMMARY (FOR SLIDES)")
    print("=" * 60)
    print(f"Total Valid Matches: {total_rows}")
    print(f"Missing Values: {null_count}")
    print(f"Average Home Goals: {home_goals_avg:.3f}")
    print(f"Average Away Goals: {away_goals_avg:.3f}")
    print(f"Home Win Rate: {(home_wins/total_rows)*100:.2f}% ({home_wins} matches)")
    print(f"Draw Rate:     {(draws/total_rows)*100:.2f}% ({draws} matches)")
    print(f"Away Win Rate: {(away_wins/total_rows)*100:.2f}% ({away_wins} matches)")

    # 80/20 Chronological Train/Test Split
    print("\n[3/4] Performing 80/20 Chronological Split...")
    split_index = int(total_rows * 0.8)
    train_df = df.iloc[:split_index].copy()
    test_df = df.iloc[split_index:].copy()

    print(f"Training Set Cutoff: First {split_index} matches (80%)")
    print(f"Testing Set Size:    Final {len(test_df)} matches (20%)")
    print(f"Training Date Range: {train_df['date'].min().strftime('%Y-%m-%d')} to {train_df['date'].max().strftime('%Y-%m-%d')}")
    print(f"Testing Date Range:  {test_df['date'].min().strftime('%Y-%m-%d')} to {test_df['date'].max().strftime('%Y-%m-%d')}")

    # Exponential Time-Decay Weighting
    print("\n[4/4] Fitting Dixon-Coles model with Time Decay (xi = 0.0065)...")
    max_date = train_df['date'].max()
    train_df['days_ago'] = (max_date - train_df['date']).dt.days
    train_df['decay_weight'] = np.exp(-0.0065 * train_df['days_ago'])

    clf = pb.models.DixonColesGoalModel(
        train_df["goals_home"],
        train_df["goals_away"],
        train_df["team_home"],
        train_df["team_away"],
        weights=train_df["decay_weight"]
    )
    clf.fit()
    print("  ✓ Model fitting successful.")

    # Calculate Out-of-Sample RPS
    rps_scores = []
    skipped_matches = 0

    for index, row in test_df.iterrows():
        try:
            probs = clf.predict(row['team_home'], row['team_away'])
            pred_h = probs.home_win
            pred_d = probs.draw

            actual_h_goals = row['goals_home']
            actual_a_goals = row['goals_away']

            if actual_h_goals > actual_a_goals:
                actual_h, actual_d = 1, 0
            elif actual_h_goals == actual_a_goals:
                actual_h, actual_d = 0, 1
            else:
                actual_h, actual_d = 0, 0

            rps = 0.5 * ((pred_h - actual_h)**2 + (pred_h + pred_d - (actual_h + actual_d))**2)
            rps_scores.append(rps)
        except ValueError:
            skipped_matches += 1

    avg_rps = np.mean(rps_scores)

    print("\n" + "=" * 60)
    print("OUT-OF-SAMPLE EVALUATION BENCHMARK")
    print("=" * 60)
    print(f"Evaluated Test Matches: {len(rps_scores)}")
    print(f"Skipped (Promoted Teams): {skipped_matches}")
    print(f"Calculated 2019-2026 Test RPS: {avg_rps:.4f}")
    print("=" * 60)

    # STAGE 2: Sequential Dynamic Bayesian Model
    # Insert this entire block, indented once, after Stage 1's final print in main().
    PROCESS_NOISE = 0.01

    from scipy.stats import poisson


    class TeamState:
        """Independent Gaussian attack and concession-tendency beliefs."""

        def __init__(self):
            self.attack_mean = 0.0
            self.attack_var = 1.0
            self.defence_mean = 0.0
            self.defence_var = 1.0


    def predict_step(state, process_noise=PROCESS_NOISE):
        state.attack_var += process_noise
        state.defence_var += process_noise
        return state


    def match_prediction_probs(home_state, away_state, home_adv, max_goals=10):
        lambda_home = np.exp(
            home_state.attack_mean + away_state.defence_mean + home_adv
        )
        mu_away = np.exp(away_state.attack_mean + home_state.defence_mean)
        if not np.isfinite([lambda_home, mu_away]).all():
            raise FloatingPointError("Non-finite pre-match goal rate.")

        goals = np.arange(max_goals + 1)
        grid = np.outer(poisson.pmf(goals, lambda_home), poisson.pmf(goals, mu_away))
        grid_mass = grid.sum()
        if not np.isfinite(grid_mass) or grid_mass <= 0:
            raise FloatingPointError("The specified score grid has no finite positive mass.")
        grid /= grid_mass

        # Rows are home goals; columns are away goals.
        pred_home_win = float(np.tril(grid, k=-1).sum())
        pred_draw = float(np.trace(grid))
        pred_away_win = float(np.triu(grid, k=1).sum())
        return pred_home_win, pred_draw, pred_away_win


    def update_step(home_state, away_state, goals_home, goals_away, lambda_home, mu_away):
        # Both updates for an observation use the SAME pre-match rate and gradient.
        home_gradient = goals_home - lambda_home
        away_gradient = goals_away - mu_away

        attack_home_precision = 1.0 / home_state.attack_var + lambda_home
        defence_away_precision = 1.0 / away_state.defence_var + lambda_home
        attack_away_precision = 1.0 / away_state.attack_var + mu_away
        defence_home_precision = 1.0 / home_state.defence_var + mu_away

        home_state.attack_mean += home_gradient / attack_home_precision
        home_state.attack_var = 1.0 / attack_home_precision
        away_state.defence_mean += home_gradient / defence_away_precision
        away_state.defence_var = 1.0 / defence_away_precision

        away_state.attack_mean += away_gradient / attack_away_precision
        away_state.attack_var = 1.0 / attack_away_precision
        home_state.defence_mean += away_gradient / defence_home_precision
        home_state.defence_var = 1.0 / defence_home_precision
        return home_state, away_state


    def run_sequential_filter(df):
        # home_adv is fixed once below from main()'s actual Stage 1 train_df.
        if not df.index.is_unique:
            raise ValueError("Unique match indices are required for exact test alignment.")
        if df[["date", "team_home", "team_away", "goals_home", "goals_away"]].isna().any().any():
            raise ValueError("Stage 2 requires complete dates, teams, and goals.")
        goals_array = df[["goals_home", "goals_away"]].to_numpy(dtype=float)
        if (not np.isfinite(goals_array).all() or (goals_array < 0).any()
                or (goals_array != np.floor(goals_array)).any()):
            raise ValueError("Poisson observations must be finite nonnegative integer goals.")
        if (df["team_home"] == df["team_away"]).any():
            raise ValueError("A match must contain two different teams.")

        # A stable sort retains Stage 1's existing order for equal timestamps.
        ordered_df = df.sort_values("date", kind="stable")
        team_states = {}
        records = []
        for row in ordered_df.itertuples():
            if row.team_home not in team_states:
                team_states[row.team_home] = TeamState()
            if row.team_away not in team_states:
                team_states[row.team_away] = TeamState()

            home_state = predict_step(team_states[row.team_home], PROCESS_NOISE)
            away_state = predict_step(team_states[row.team_away], PROCESS_NOISE)
            lambda_home = np.exp(home_state.attack_mean + away_state.defence_mean + home_adv)
            mu_away = np.exp(away_state.attack_mean + home_state.defence_mean)
            pred_h, pred_d, pred_a = match_prediction_probs(home_state, away_state, home_adv)

            # Save the forecast before incorporating this match's observed score.
            records.append({
                "date": row.date,
                "team_home": row.team_home,
                "team_away": row.team_away,
                "goals_home": row.goals_home,
                "goals_away": row.goals_away,
                "pred_home_win": pred_h,
                "pred_draw": pred_d,
                "pred_away_win": pred_a,
            })
            home_state, away_state = update_step(
                home_state, away_state, row.goals_home, row.goals_away, lambda_home, mu_away
            )
            team_states[row.team_home] = home_state
            team_states[row.team_away] = away_state

        columns = ["date", "team_home", "team_away", "goals_home", "goals_away",
                   "pred_home_win", "pred_draw", "pred_away_win"]
        return pd.DataFrame(records, index=ordered_df.index, columns=columns), team_states


    print("\n" + "=" * 60)
    print("STAGE 2: SEQUENTIAL DYNAMIC BAYESIAN FILTER")
    print("=" * 60)
    if train_df.empty or test_df.empty:
        raise ValueError("Both Stage 1 training and test sets must be nonempty.")
    train_home_mean = train_df["goals_home"].mean()
    train_away_mean = train_df["goals_away"].mean()
    if (not np.isfinite([train_home_mean, train_away_mean]).all()
            or train_home_mean <= 0 or train_away_mean <= 0):
        raise ValueError("Training goal means must be finite and positive for home_adv.")
    home_adv = float(np.log(train_home_mean) - np.log(train_away_mean))
    print(f"Fixed training-only home_adv: {home_adv!r}")
    print(f"Constant process noise per team appearance: {PROCESS_NOISE}")
    stage2_predictions, final_team_states = run_sequential_filter(df)

    # Intersect the requested date range with the actual held-out row IDs.
    # Date-only filtering could include training matches on the split date.
    test_date_mask = stage2_predictions["date"].between(
        test_df["date"].min(), test_df["date"].max(), inclusive="both"
    )
    stage2_test = stage2_predictions.loc[
        test_date_mask & stage2_predictions.index.isin(test_df.index)
    ].loc[test_df.index]
    pd.testing.assert_frame_equal(
        stage2_test[["date", "team_home", "team_away", "goals_home", "goals_away"]],
        test_df[["date", "team_home", "team_away", "goals_home", "goals_away"]],
        check_dtype=False,
    )


    def stage2_rps(rows):
        scores = []
        for row in rows.itertuples():
            pred_h, pred_d = row.pred_home_win, row.pred_draw
            actual_h_goals, actual_a_goals = row.goals_home, row.goals_away
            if actual_h_goals > actual_a_goals:
                actual_h, actual_d = 1, 0
            elif actual_h_goals == actual_a_goals:
                actual_h, actual_d = 0, 1
            else:
                actual_h, actual_d = 0, 0
            rps = 0.5 * ((pred_h - actual_h)**2 + (pred_h + pred_d - (actual_h + actual_d))**2)
            scores.append(rps)
        if not scores:
            raise ValueError("No matches are available for the requested RPS comparison.")
        return float(np.mean(scores))


    stage2_all_test_rps = stage2_rps(stage2_test)

    # Stage 1 skips unseen teams. Restrict the primary comparison to those same
    # matches, without fitting or invoking any Stage 1 model in the dynamic filter.
    # Verify the count so an unrelated baseline ValueError cannot go unnoticed.
    stage1_known_teams = set(train_df["team_home"]) | set(train_df["team_away"])
    common_mask = (stage2_test["team_home"].isin(stage1_known_teams)
                   & stage2_test["team_away"].isin(stage1_known_teams))
    stage2_common_test = stage2_test.loc[common_mask]
    if len(stage2_common_test) != len(rps_scores):
        raise ValueError(
            "Known-team test rows do not match Stage 1's evaluated count; "
            "inspect its skipped matches before claiming an identical-match comparison."
        )
    stage2_common_rps = stage2_rps(stage2_common_test)

    # Additional metrics on the exact same rows used for the RPS comparison.
    def brier_and_log_loss(predicted_probs, actual_one_hot):
        # Mean squared error across all three classes, then across matches.
        brier = float(np.mean((predicted_probs - actual_one_hot) ** 2))
        actual_probs = np.sum(predicted_probs * actual_one_hot, axis=1)
        log_loss = float(-np.mean(np.log(np.clip(actual_probs, 1e-15, 1.0))))
        return brier, log_loss


    common_goals_home = stage2_common_test["goals_home"].to_numpy()
    common_goals_away = stage2_common_test["goals_away"].to_numpy()
    common_actual_class = np.where(
        common_goals_home > common_goals_away, 0,
        np.where(common_goals_home == common_goals_away, 1, 2)
    )
    common_one_hot = np.eye(3)[common_actual_class]  # Home, draw, away.

    stage1_common_probs = []
    for row in stage2_common_test.itertuples():
        stage1_match_probs = clf.predict(row.team_home, row.team_away)
        stage1_common_probs.append([
            stage1_match_probs.home_win, stage1_match_probs.draw, stage1_match_probs.away_win
        ])
    stage1_common_probs = np.asarray(stage1_common_probs, dtype=float)
    stage2_common_probs = stage2_common_test[
        ["pred_home_win", "pred_draw", "pred_away_win"]
    ].to_numpy(dtype=float)
    stage1_brier, stage1_log_loss = brier_and_log_loss(stage1_common_probs, common_one_hot)
    stage2_brier, stage2_log_loss = brier_and_log_loss(stage2_common_probs, common_one_hot)

    print("\n" + "=" * 60)
    print(f"IDENTICAL-MATCH COMPARISON ({len(stage2_common_test)} matches)")
    print("=" * 60)
    print(f"{'':<10}{'RPS':>10}{'Brier':>10}{'LogLoss':>10}")
    print(f"{'Stage 1:':<10}{avg_rps:>10.4f}{stage1_brier:>10.4f}{stage1_log_loss:>10.4f}")
    print(f"{'Stage 2:':<10}{stage2_common_rps:>10.4f}{stage2_brier:>10.4f}{stage2_log_loss:>10.4f}")
    print(f"Stage 2 RPS on ALL {len(stage2_test)} held-out matches: {stage2_all_test_rps:.4f}")
    print(f"Stage 1 skipped {len(stage2_test) - len(stage2_common_test)} of those matches.")
    print("Stage 2 forecasts each match before updating, including during the test period.")

    club_aliases = [
        ("Man City", {"man city", "manchester city", "manchester city fc"}),
        ("Real Madrid", {"real madrid", "real madrid cf"}),
        ("Bayern Munich", {"bayern munich", "bayern munchen", "bayern münchen", "bayern", "fc bayern munich"}),
        ("Paris SG", {"paris sg", "paris saint germain", "paris saint-germain", "psg"}),
        ("Juventus", {"juventus", "juventus fc"}),
    ]
    print("\nFINAL TEAM STATES (higher defence_mean means more concessions)")
    home_team_names = df["team_home"].unique()
    for club_label, aliases in club_aliases:
        matches = [name for name in home_team_names
                   if " ".join(str(name).casefold().replace("_", " ").replace("-", " ").split())
                   in aliases]
        if not matches:
            print(f"{club_label}: not present under the recognized team-name aliases")
        for name in matches:
            state = final_team_states[name]
            print(f"{name}: attack_mean={state.attack_mean!r}, defence_mean={state.defence_mean!r}, "
                  f"attack_var={state.attack_var!r}, defence_var={state.defence_var!r}")
    print("=" * 60)

    print('STAGE 3 (RESEARCH MILESTONE): PERSISTENT CHANGE DETECTOR — CONSERVATIVE DESIGN, VALIDATED NULL RESULT')
    # One ordinary Poisson update per match. Surprise never scales its gradient
    # or Fisher information; persistent evidence can increase FUTURE process noise.
    CHANGE_SCALE_CANDIDATES = (0.0, 0.25, 0.5, 1.0, 2.0)
    PREDICTION_LEAD_MINUTES = 60
    LINEUP_DELTA_LIMIT = 0.5


    def prediction_surprise_score(pred_home_win, pred_draw, pred_away_win,
                                  goals_home, goals_away):
        p_actual = (pred_home_win if goals_home > goals_away else
                    pred_draw if goals_home == goals_away else pred_away_win)
        probs = np.array([pred_home_win, pred_draw, pred_away_win], dtype=float)
        entropy = -np.sum(probs * np.log(probs + 1e-9))
        return float(np.clip(-np.log(max(p_actual, 1e-6)) / (entropy + 1e-6), 0.2, 5.0))


    class ChangeEvidence:
        """Bounded, signed evidence for one team's attack OR concession tendency."""
        def __init__(self):
            self.ewma = 0.0
            self.direction = 0
            self.run = 0
            self.count = 0
            self.recent_surprises = []
            self.boost = 0.0

        def observe(self, goals, rate, surprise):
            # Detector statistic only; never an extra parameter update.
            z = float(np.clip((goals - rate) / np.sqrt(max(rate, 1e-6)), -3.0, 3.0))
            self.ewma = 0.75 * self.ewma + 0.25 * z
            direction = int(np.sign(z)) if abs(z) >= 0.5 else 0
            self.run = self.run + 1 if direction != 0 and direction == self.direction else int(direction != 0)
            self.direction = direction
            self.count += 1
            self.recent_surprises = (self.recent_surprises + [surprise >= 1.5])[-5:]
            persistent = (self.count >= 5 and self.run >= 3
                          and sum(self.recent_surprises) >= 2
                          and abs(self.ewma) >= 0.5)
            self.boost = min(1.0, abs(self.ewma) / 2.0) if persistent else 0.5 * self.boost
            return persistent


    def prepare_lineup_offsets(matches, adjustments=None):
        """Precomputed player-based log-rate deltas, keyed to real fixtures.

        Define LINEUP_ADJUSTMENTS as a DataFrame BEFORE running this model cell.
        Required: date, competition, team_home, team_away, available_at,
        ratings_asof, home_attack_delta, home_defence_delta, away_attack_delta,
        away_defence_delta. Defence effects use the concession-tendency sign.
        Effects must be estimated from past data. Timestamp checks cannot certify
        how external ratings were trained. All timestamps must use the fixture
        clock; naive values are interpreted as UTC. Normalize source clocks first.
        """
        result = pd.DataFrame(0.0, index=matches.index, columns=[
            'home_attack_delta', 'home_defence_delta', 'away_attack_delta', 'away_defence_delta'])
        result['lineup_available'] = False
        if adjustments is None or adjustments.empty:
            return result
        effects = list(result.columns[:4])
        keys = ['date', 'competition', 'team_home', 'team_away']
        required = keys + ['available_at', 'ratings_asof'] + effects
        if not set(required).issubset(adjustments.columns):
            raise ValueError('Lineup data missing columns: ' + str(sorted(set(required) - set(adjustments.columns))))
        supplied = adjustments[required].copy()
        if supplied.isna().any().any():
            raise ValueError('Lineup keys, timestamps and effects must be complete.')
        fixture_keys = matches[keys].copy()
        fixture_keys['date'] = pd.to_datetime(fixture_keys.date, utc=True)
        for name in ('date', 'available_at', 'ratings_asof'):
            supplied[name] = pd.to_datetime(supplied[name], utc=True, errors='raise')
            if supplied[name].isna().any():
                raise ValueError('Lineup timestamps must not parse to NaT.')
        if fixture_keys.duplicated(keys).any() or supplied.duplicated(keys).any():
            raise ValueError('Fixture and lineup keys must be unique; select one as-of snapshot per fixture.')
        values = supplied[effects].to_numpy(dtype=float)
        if not np.isfinite(values).all() or (np.abs(values) > LINEUP_DELTA_LIMIT).any():
            raise ValueError('Lineup effects must be finite and bounded by LINEUP_DELTA_LIMIT.')
        cutoff = supplied.date - pd.Timedelta(minutes=PREDICTION_LEAD_MINUTES)
        if (supplied.available_at > cutoff).any() or (supplied.ratings_asof > cutoff).any():
            raise ValueError('Lineup/rating data was unavailable at the forecast cutoff.')
        fixture_index = pd.MultiIndex.from_frame(fixture_keys)
        positions = fixture_index.get_indexer(pd.MultiIndex.from_frame(supplied[keys]))
        if (positions < 0).any():
            raise ValueError('Lineup data contains an unmatched date/competition/team fixture key.')
        result.iloc[positions, :4] = values
        result.iloc[positions, 4] = True
        return result


    def run_change_filter(matches, change_scale=0.0, home_adv_override=None, lineup_offsets=None):
        if not np.isfinite(change_scale) or not 0.0 <= change_scale <= 2.0:
            raise ValueError('change_scale must be finite and in [0, 2].')
        advantage = home_adv if home_adv_override is None else float(home_adv_override)
        if not np.isfinite(advantage):
            raise ValueError('Home advantage must be finite.')
        if not matches.index.is_unique:
            raise ValueError('Unique match indices are required.')
        required = ['date', 'team_home', 'team_away', 'goals_home', 'goals_away']
        if matches[required].isna().any().any():
            raise ValueError('Incomplete match data.')
        goals = matches[['goals_home', 'goals_away']].to_numpy(dtype=float)
        if not np.isfinite(goals).all() or (goals < 0).any() or (goals != np.floor(goals)).any():
            raise ValueError('Goals must be finite nonnegative integers.')
        if (matches.team_home == matches.team_away).any():
            raise ValueError('A team cannot play itself.')
        ordered = matches.sort_values('date', kind='stable')
        if lineup_offsets is None:
            offsets = np.zeros((len(ordered), 4))
            available = np.zeros(len(ordered), bool)
        else:
            aligned = lineup_offsets.loc[ordered.index]
            offsets = aligned[['home_attack_delta', 'home_defence_delta',
                               'away_attack_delta', 'away_defence_delta']].to_numpy(float)
            available = aligned.lineup_available.to_numpy(bool)
            if not np.isfinite(offsets).all() or (np.abs(offsets) > LINEUP_DELTA_LIMIT).any():
                raise ValueError('Invalid prepared lineup offsets.')
        states, evidence, records = {}, {}, []
        for position, row in enumerate(ordered.itertuples()):
            for team in (row.team_home, row.team_away):
                if team not in states:
                    states[team] = TeamState()
                    evidence[team] = (ChangeEvidence(), ChangeEvidence())
            hs, aws = states[row.team_home], states[row.team_away]
            ha, hd = evidence[row.team_home]
            aa, ad = evidence[row.team_away]
            prior_boosts = np.array([ha.boost, hd.boost, aa.boost, ad.boost])
            # Future-prior variance: no mean shift, no same-result feedback.
            extra = PROCESS_NOISE * change_scale * prior_boosts
            predict_step(hs, PROCESS_NOISE)
            predict_step(aws, PROCESS_NOISE)
            hs.attack_var += extra[0]
            hs.defence_var += extra[1]
            aws.attack_var += extra[2]
            aws.defence_var += extra[3]
            delta = offsets[position]
            if not np.any(delta):
                prediction_home, prediction_away = hs, aws
            else:
                prediction_home, prediction_away = TeamState(), TeamState()
                prediction_home.attack_mean = hs.attack_mean + delta[0]
                prediction_home.defence_mean = hs.defence_mean + delta[1]
                prediction_away.attack_mean = aws.attack_mean + delta[2]
                prediction_away.defence_mean = aws.defence_mean + delta[3]
            lambda_home = np.exp(prediction_home.attack_mean + prediction_away.defence_mean + advantage)
            mu_away = np.exp(prediction_away.attack_mean + prediction_home.defence_mean)
            pred_h, pred_d, pred_a = match_prediction_probs(prediction_home, prediction_away, advantage)
            surprise = prediction_surprise_score(pred_h, pred_d, pred_a, row.goals_home, row.goals_away)
            records.append(dict(date=row.date, team_home=row.team_home, team_away=row.team_away,
                                goals_home=row.goals_home, goals_away=row.goals_away,
                                pred_home_win=pred_h, pred_draw=pred_d, pred_away_win=pred_a,
                                surprise_score=surprise, extra_process_variance=float(extra.sum()),
                                max_extra_component_variance=float(extra.max()),
                                lineup_available=bool(available[position])))
            # The ONLY observation update: ordinary unscaled Poisson gradient/Fisher.
            update_step(hs, aws, row.goals_home, row.goals_away, lambda_home, mu_away)
            # Known lineup context is included in residuals; evidence affects later priors.
            ha.observe(row.goals_home, lambda_home, surprise)
            ad.observe(row.goals_home, lambda_home, surprise)
            aa.observe(row.goals_away, mu_away, surprise)
            hd.observe(row.goals_away, mu_away, surprise)
        return pd.DataFrame(records, index=ordered.index), states


    def probability_metrics(probabilities, actual_class):
        p = np.asarray(probabilities, float)
        if p.shape != (len(actual_class), 3) or not np.isfinite(p).all() or (p < 0).any():
            raise ValueError('Invalid Home/Draw/Away probability matrix.')
        if not np.allclose(p.sum(axis=1), 1.0, atol=1e-10):
            raise ValueError('Probabilities must sum to one.')
        actual = np.eye(3)[actual_class]
        rps = float(np.mean((np.cumsum(p, axis=1)[:, :2] - np.cumsum(actual, axis=1)[:, :2]) ** 2))
        brier, loss = brier_and_log_loss(p, actual)
        return dict(RPS=rps, Brier=brier, LogLoss=loss,
                    Accuracy=float(np.mean(p.argmax(axis=1) == actual_class)))


    def outcome_classes(rows):
        return np.where(rows.goals_home > rows.goals_away, 0,
                        np.where(rows.goals_home == rows.goals_away, 1, 2))


    def select_change_scale(training_rows):
        cuts = [int(len(training_rows) * value) for value in (.55, .70, .85, 1.)]
        if cuts[0] == 0 or any(b <= a for a, b in zip(cuts[:-1], cuts[1:])):
            raise ValueError('Insufficient chronological training data.')
        records = []
        cols = ['pred_home_win', 'pred_draw', 'pred_away_win']
        for fold, (start, end) in enumerate(zip(cuts[:-1], cuts[1:]), 1):
            prefix = training_rows.iloc[:start]
            advantage = float(np.log(prefix.goals_home.mean()) - np.log(prefix.goals_away.mean()))
            actual = training_rows.iloc[start:end]
            for scale in CHANGE_SCALE_CANDIDATES:
                forecasts, _ = run_change_filter(training_rows.iloc[:end], scale, advantage)
                values = probability_metrics(forecasts.loc[actual.index, cols], outcome_classes(actual))
                records.append(dict(fold=fold, scale=scale, n=len(actual), home_adv=advantage,
                                    start=start, end=end, **values))
            print(f'  Change-detector validation fold {fold}/3 complete ({end-start} matches).')
        folds = pd.DataFrame(records)
        names = ['RPS', 'Brier', 'LogLoss', 'Accuracy']
        summary = []
        for scale in CHANGE_SCALE_CANDIDATES:
            rows = folds.loc[folds.scale == scale]
            values = np.average(rows[names], axis=0, weights=rows.n)
            summary.append(dict(scale=scale, **dict(zip(names, values))))
        summary = pd.DataFrame(summary)
        base = summary.iloc[0]
        eligible = (summary[['RPS', 'Brier', 'LogLoss']] <= base[['RPS', 'Brier', 'LogLoss']] + 1e-12).all(axis=1)
        eligible &= summary.Accuracy >= base.Accuracy - 1e-12
        summary['eligible'] = eligible
        best = summary.loc[eligible].sort_values(['RPS', 'scale'], kind='stable').iloc[0]
        return float(best.scale), summary, folds


    def calibration_summary(probabilities, labels):
        records = []
        for cls, name in enumerate(('Home', 'Draw', 'Away')):
            p = np.asarray(probabilities)[:, cls]
            y = (labels == cls).astype(float)
            bins = np.minimum((p * 10).astype(int), 9)
            for group in range(10):
                mask = bins == group
                if mask.any():
                    records.append(dict(outcome=name, bin=group, n=int(mask.sum()),
                                        forecast=float(p[mask].mean()), observed=float(y[mask].mean())))
        return pd.DataFrame(records)


    print('\n' + '=' * 70)
    print('STAGE 3: PERSISTENT CHANGE DETECTOR; SINGLE POISSON UPDATE')
    print('=' * 70)
    CHANGE_SCALE, change_validation, change_validation_folds = select_change_scale(train_df)
    print(change_validation.to_string(index=False, float_format=lambda x: f'{x:.7f}'))
    print(f'Selected CHANGE_SCALE: {CHANGE_SCALE:g}')
    if CHANGE_SCALE == 0.0:
        print('Validation selected ordinary process noise; detector scores are still recorded.')
        print('This conservative design is retained as a robustness milestone, not the active mechanism.')
    else:
        print(f'Per-component process variance is bounded between {PROCESS_NOISE:g} and {PROCESS_NOISE*(1+CHANGE_SCALE):g}.')
    print('Each match uses exactly one ordinary, unscaled observation update.')
    print('Evaluation is retrospective: this test period has been examined previously.')

    stage3_predictions, stage3_final_team_states = run_change_filter(df, CHANGE_SCALE)
    stage3_test = stage3_predictions.loc[test_df.index]
    stage3_common_test = stage3_test.loc[stage2_common_test.index]
    match_columns = ['date', 'team_home', 'team_away', 'goals_home', 'goals_away']
    pd.testing.assert_frame_equal(stage3_common_test[match_columns], stage2_common_test[match_columns], check_dtype=False)
    prob_columns = ['pred_home_win', 'pred_draw', 'pred_away_win']
    stage3_common_probs = stage3_common_test[prob_columns].to_numpy(float)
    stage3_metrics = probability_metrics(stage3_common_probs, common_actual_class)
    stage3_common_rps, stage3_brier, stage3_log_loss = [stage3_metrics[key] for key in ('RPS', 'Brier', 'LogLoss')]
    comparison = pd.DataFrame([
        dict(Model='Stage 1', **probability_metrics(stage1_common_probs, common_actual_class)),
        dict(Model='Stage 2', **probability_metrics(stage2_common_probs, common_actual_class)),
        dict(Model='Stage 3', **stage3_metrics),
    ])
    print(f'\nIDENTICAL-MATCH COMPARISON ({len(stage3_common_test)} matches)')
    print(comparison.to_string(index=False, float_format=lambda x: f'{x:.7f}'))
    print(f'Test matches with extra variance: {(stage3_test.extra_process_variance > 0).sum()}/{len(stage3_test)}')
    print(f'Raw surprise mean/max: {stage3_test.surprise_score.mean():.4f}/{stage3_test.surprise_score.max():.4f}')
    print(f'Max extra variance per component: {stage3_test.max_extra_component_variance.max():.6f}')

    # Optional real lineup integration. Missing data is explicitly unevaluated.
    lineup_input = globals().get('LINEUP_ADJUSTMENTS', None)
    lineup_offsets = prepare_lineup_offsets(df, lineup_input)
    ablation = [dict(Model='Stage 2 baseline', Status='evaluated', **comparison.iloc[1].drop('Model').to_dict()),
                dict(Model='Baseline + change detector', Status='evaluated', **stage3_metrics)]
    lineup_predictions = combined_predictions = None
    if lineup_offsets.lineup_available.any():
        lineup_predictions, _ = run_change_filter(df, 0.0, lineup_offsets=lineup_offsets)
        combined_predictions, _ = run_change_filter(df, CHANGE_SCALE, lineup_offsets=lineup_offsets)
        for name, forecasts in [('Baseline + lineup', lineup_predictions),
                                 ('Baseline + lineup + detector', combined_predictions)]:
            values = probability_metrics(forecasts.loc[stage2_common_test.index, prob_columns], common_actual_class)
            ablation.append(dict(Model=name, Status='evaluated; partial lineup coverage possible', **values))
        covered = lineup_offsets.loc[stage2_common_test.index].lineup_available
        print(f'Lineup coverage on common test: {covered.sum()}/{len(covered)} at T-{PREDICTION_LEAD_MINUTES} minutes.')
        print('Detector scale is frozen from no-lineup validation; joint retuning remains separate.')
    else:
        for name in ('Baseline + lineup', 'Baseline + lineup + detector'):
            ablation.append(dict(Model=name, Status='NOT EVALUATED: no historical lineup data'))
    ablation = pd.DataFrame(ablation)
    print('\nFOUR-WAY ABLATION')
    print(ablation.to_string(index=False, float_format=lambda x: f'{x:.7f}'))

    classification_results, calibration_results = {}, {}
    print('\nOUTCOME CLASSIFICATION AND CALIBRATION (Home / Draw / Away)')
    for name, p in [('Stage 1', stage1_common_probs), ('Stage 2', stage2_common_probs), ('Stage 3', stage3_common_probs)]:
        confusion = np.zeros((3, 3), dtype=int)
        np.add.at(confusion, (common_actual_class, p.argmax(axis=1)), 1)
        recall = np.divide(np.diag(confusion), confusion.sum(axis=1), out=np.zeros(3), where=confusion.sum(axis=1) > 0)
        classification_results[name] = dict(confusion=confusion, recall=recall)
        bins = calibration_summary(p, common_actual_class)
        calibration_results[name] = bins
        ece = {label: float(np.average(np.abs(g.forecast-g.observed), weights=g.n))
               for label, g in bins.groupby('outcome')}
        draw = (common_actual_class == 1).astype(float)
        draw_p = np.clip(p[:, 1], 1e-15, 1-1e-15)
        print(f'{name}: draw probability={p[:,1].mean():.4f}; actual draw rate={draw.mean():.4f}; '
              f'draw Brier={np.mean((p[:,1]-draw)**2):.4f}; '
              f'draw binary log-loss={-np.mean(draw*np.log(draw_p)+(1-draw)*np.log(1-draw_p)):.4f}')
        print('  Class calibration ECE:', ece)
        print('  Recall H/D/A:', recall, '\n  Confusion matrix (actual rows):\n', confusion)
    print('\nStage 3 reliability bins:')
    print(calibration_results['Stage 3'].to_string(index=False, float_format=lambda x: f'{x:.4f}'))

    # Closing odds: evaluation only, with later information than T-60 forecasts.
    # Source: https://www.football-data.co.uk/data (C columns denote closing odds).
    market_columns = ['avg_ch', 'avg_cd', 'avg_ca']
    market_comparison = None
    if set(market_columns).issubset(df.columns):
        odds = df.loc[stage2_common_test.index, market_columns].apply(pd.to_numeric, errors='coerce').to_numpy()
        market_mask = np.isfinite(odds).all(axis=1) & (odds > 1.0).all(axis=1)
        if market_mask.any():
            implied = 1.0 / odds[market_mask]
            market_probs = implied / implied.sum(axis=1, keepdims=True)
            market_comparison = pd.DataFrame([
                dict(Model='Market closing average (normalized)', **probability_metrics(market_probs, common_actual_class[market_mask])),
                dict(Model='Stage 2 on identical market rows', **probability_metrics(stage2_common_probs[market_mask], common_actual_class[market_mask])),
                dict(Model='Stage 3 on identical market rows', **probability_metrics(stage3_common_probs[market_mask], common_actual_class[market_mask])),
            ])
            print(f'\nCLOSING-MARKET BENCHMARK: {market_mask.sum()} identical rows; later-information comparison')
            print(market_comparison.to_string(index=False, float_format=lambda x: f'{x:.7f}'))
        else:
            print('Market benchmark unavailable: no valid closing-odds triples.')
    else:
        print('Market benchmark unavailable: closing-odds columns missing.')

    # Paired week-block intervals are never used for model selection.
    one_hot = np.eye(3)[common_actual_class]
    rps2_rows = np.mean((np.cumsum(stage2_common_probs, axis=1)[:, :2] - np.cumsum(one_hot, axis=1)[:, :2])**2, axis=1)
    rps3_rows = np.mean((np.cumsum(stage3_common_probs, axis=1)[:, :2] - np.cumsum(one_hot, axis=1)[:, :2])**2, axis=1)
    week_ids = stage3_common_test.date.dt.to_period('W').astype(str).to_numpy()
    weekly = pd.DataFrame(dict(week=week_ids, delta=rps3_rows-rps2_rows)).groupby('week').delta.agg(['sum', 'count'])
    rng = np.random.default_rng(20260922)
    resamples = rng.integers(len(weekly), size=(2000, len(weekly)))
    bootstrap_delta = weekly['sum'].to_numpy()[resamples].sum(axis=1) / weekly['count'].to_numpy()[resamples].sum(axis=1)
    stage3_rps_delta_ci = np.quantile(bootstrap_delta, [.025, .975])
    print('\nStage 3 minus Stage 2 RPS: ', float(np.mean(rps3_rows-rps2_rows)))
    print('95% paired week-block bootstrap interval: ', stage3_rps_delta_ci)

    stage2_test_surprise = np.asarray([prediction_surprise_score(row.pred_home_win, row.pred_draw,
        row.pred_away_win, row.goals_home, row.goals_away) for row in stage2_test.itertuples()])
    top_n = max(1, int(np.ceil(.1 * len(stage2_test))))
    top_positions = np.argsort(-stage2_test_surprise, kind='stable')[:top_n]
    print(f'\nTop 10% most surprising matches (N={top_n}); retrospective subgroup, not causal evidence:')
    print(f'  Stage 2 RPS: {stage2_rps(stage2_test.iloc[top_positions]):.7f}')
    print(f'  Stage 3 RPS: {stage2_rps(stage3_test.iloc[top_positions]):.7f}')
    post_upset_results = []
    for horizon in (1, 5, 10):
        positions = set()
        for pos in top_positions:
            row = test_df.iloc[pos]
            later = test_df.iloc[pos+1:]
            affected = later.team_home.isin([row.team_home, row.team_away]) | later.team_away.isin([row.team_home, row.team_away])
            positions.update(later.index[affected][:horizon])
        ids = sorted(positions)
        if ids:
            post_upset_results.append(dict(next_team_appearances=horizon, n=len(ids),
                                           Stage2_RPS=stage2_rps(stage2_test.loc[ids]),
                                           Stage3_RPS=stage2_rps(stage3_test.loc[ids])))
    print('\nACTUAL SUBSEQUENT MATCHES AFTER UPSETS (deduplicated):')
    print(pd.DataFrame(post_upset_results).to_string(index=False, float_format=lambda x: f'{x:.7f}'))
    print('=' * 70)
    # STAGE 4: CURRENT-MATCH SURPRISE-SCALED FISHER INFORMATION
    # Append-only experiment: larger multipliers DAMP the current mean correction.
    # They also shrink posterior variances more; this is not gradient amplification.
    SENSITIVITY_CANDIDATES = (0.0, 0.25, 0.5, 1.0)


    def surprise_to_update_mult(raw_surprise, sensitivity):
        if not np.isfinite(sensitivity) or not 0.0 <= sensitivity <= 1.0:
            raise ValueError('sensitivity must be finite and in [0, 1].')
        if not np.isfinite(raw_surprise):
            raise ValueError('raw_surprise must be finite.')
        mult = 1.0 + sensitivity * (raw_surprise - 1.0)
        return float(np.clip(mult, 0.5, 2.0))


    def update_step_with_surprise(home_state, away_state, goals_home, goals_away,
                                  lambda_home, mu_away, surprise_mult=1.0):
        if not np.isfinite(surprise_mult) or not 0.5 <= surprise_mult <= 2.0:
            raise ValueError('surprise_mult must be finite and in [0.5, 2.0].')
        home_gradient = goals_home - lambda_home
        away_gradient = goals_away - mu_away
        attack_home_precision = 1.0 / home_state.attack_var + lambda_home * surprise_mult
        defence_away_precision = 1.0 / away_state.defence_var + lambda_home * surprise_mult
        attack_away_precision = 1.0 / away_state.attack_var + mu_away * surprise_mult
        defence_home_precision = 1.0 / home_state.defence_var + mu_away * surprise_mult

        home_state.attack_mean += home_gradient / attack_home_precision
        home_state.attack_var = 1.0 / attack_home_precision
        away_state.defence_mean += home_gradient / defence_away_precision
        away_state.defence_var = 1.0 / defence_away_precision
        away_state.attack_mean += away_gradient / attack_away_precision
        away_state.attack_var = 1.0 / attack_away_precision
        home_state.defence_mean += away_gradient / defence_home_precision
        home_state.defence_var = 1.0 / defence_home_precision
        return home_state, away_state


    def run_sequential_filter_with_surprise(df, sensitivity, *, home_adv_override=None):
        # The optional override keeps validation home advantage training-prefix-only.
        # It never modifies Stage 2's captured home_adv or any existing team state.
        surprise_to_update_mult(1.0, sensitivity)
        advantage = home_adv if home_adv_override is None else float(home_adv_override)
        if not np.isfinite(advantage):
            raise ValueError('home advantage must be finite.')
        if not df.index.is_unique:
            raise ValueError('Unique match indices are required for exact test alignment.')
        required = ['date', 'team_home', 'team_away', 'goals_home', 'goals_away']
        if df[required].isna().any().any():
            raise ValueError('Stage 4 requires complete dates, teams, and goals.')
        goals_array = df[['goals_home', 'goals_away']].to_numpy(dtype=float)
        if (not np.isfinite(goals_array).all() or (goals_array < 0).any()
                or (goals_array != np.floor(goals_array)).any()):
            raise ValueError('Poisson observations must be finite nonnegative integer goals.')
        if (df.team_home == df.team_away).any():
            raise ValueError('A match must contain two different teams.')
        ordered_df = df.sort_values('date', kind='stable')
        team_states, records = {}, []
        for row in ordered_df.itertuples():
            if row.team_home not in team_states:
                team_states[row.team_home] = TeamState()
            if row.team_away not in team_states:
                team_states[row.team_away] = TeamState()
            home_state = predict_step(team_states[row.team_home], PROCESS_NOISE)
            away_state = predict_step(team_states[row.team_away], PROCESS_NOISE)
            lambda_home = np.exp(home_state.attack_mean + away_state.defence_mean + advantage)
            mu_away = np.exp(away_state.attack_mean + home_state.defence_mean)
            pred_h, pred_d, pred_a = match_prediction_probs(home_state, away_state, advantage)
            # Freeze the forecast before consulting the result or computing surprise.
            records.append(dict(date=row.date, team_home=row.team_home, team_away=row.team_away,
                                goals_home=row.goals_home, goals_away=row.goals_away,
                                pred_home_win=pred_h, pred_draw=pred_d, pred_away_win=pred_a))
            raw_surprise = prediction_surprise_score(pred_h, pred_d, pred_a, row.goals_home, row.goals_away)
            mult = surprise_to_update_mult(raw_surprise, sensitivity)
            records[-1]['surprise_mult'] = mult
            home_state, away_state = update_step_with_surprise(
                home_state, away_state, row.goals_home, row.goals_away, lambda_home, mu_away, mult)
            team_states[row.team_home], team_states[row.team_away] = home_state, away_state
        columns = required + ['pred_home_win', 'pred_draw', 'pred_away_win', 'surprise_mult']
        return pd.DataFrame(records, index=ordered_df.index, columns=columns), team_states


    def select_stage4_sensitivity(training_rows, existing_fold_rows):
        # Reuse Stage 3's already computed boundaries, not its multi-metric gate.
        plan = existing_fold_rows[['fold', 'start', 'end', 'home_adv']].drop_duplicates().sort_values('fold')
        if len(plan) != 3 or not plan.fold.is_unique:
            raise ValueError('Expected the three unique existing expanding folds.')
        fold_results = []
        held_out = {s: [] for s in SENSITIVITY_CANDIDATES}
        for fold in plan.itertuples(index=False):
            start, end = int(fold.start), int(fold.end)
            if not 0 < start < end <= len(training_rows):
                raise ValueError('Validation fold leaves the training dataset.')
            prefix = training_rows.iloc[:start]
            expected_adv = float(np.log(prefix.goals_home.mean()) - np.log(prefix.goals_away.mean()))
            if not np.isclose(expected_adv, fold.home_adv, rtol=0, atol=1e-14):
                raise ValueError('Validation home advantage must use only its training prefix.')
            actual = training_rows.iloc[start:end]
            for sensitivity in SENSITIVITY_CANDIDATES:
                # Warm up on the prefix, then forecast/update online on its held-out slice.
                forecasts, _ = run_sequential_filter_with_surprise(
                    training_rows.iloc[:end], sensitivity, home_adv_override=fold.home_adv)
                held = forecasts.loc[actual.index]
                values = probability_metrics(held[prob_columns], outcome_classes(actual))
                held_out[sensitivity].append(held)
                fold_results.append(dict(fold=int(fold.fold), sensitivity=sensitivity, n=len(actual),
                                         start=start, end=end, home_adv=float(fold.home_adv), **values))
            print(f'Stage 4 validation fold {fold.fold}: prefix={start}, held-out={end-start} complete.', flush=True)
        folds = pd.DataFrame(fold_results)
        summaries = []
        for sensitivity in SENSITIVITY_CANDIDATES:
            rows = folds.loc[folds.sensitivity == sensitivity]
            summaries.append(dict(sensitivity=sensitivity, **{
                key: float(np.average(rows[key], weights=rows.n))
                for key in ('RPS', 'Brier', 'LogLoss', 'Accuracy')}))
        summary = pd.DataFrame(summaries)
        # Stable candidate order breaks exact ties in favor of smaller sensitivity.
        selected = float(summary.loc[summary.LogLoss.idxmin(), 'sensitivity'])
        pooled = {s: pd.concat(parts) for s, parts in held_out.items()}
        if any(not rows.index.is_unique for rows in pooled.values()):
            raise ValueError('Validation observations must not be counted twice.')
        return selected, summary, folds, pooled


    def stage4_logloss_interval(selected_rows, baseline_rows):
        pd.testing.assert_frame_equal(selected_rows[match_columns], baseline_rows[match_columns])
        labels = outcome_classes(baseline_rows)
        positions = np.arange(len(labels))
        selected_p = selected_rows[prob_columns].to_numpy(float)
        baseline_p = baseline_rows[prob_columns].to_numpy(float)
        delta = (-np.log(np.clip(selected_p[positions, labels], 1e-15, 1.0))
                 + np.log(np.clip(baseline_p[positions, labels], 1e-15, 1.0)))
        weekly = pd.DataFrame(dict(week=baseline_rows.date.dt.to_period('W').astype(str).to_numpy(),
                                  delta=delta)).groupby('week').delta.agg(['sum', 'count'])
        rng = np.random.default_rng(20260927)
        resamples = rng.integers(len(weekly), size=(2000, len(weekly)))
        bootstrap = (weekly['sum'].to_numpy()[resamples].sum(axis=1)
                     / weekly['count'].to_numpy()[resamples].sum(axis=1))
        ci = np.quantile(bootstrap, [.025, .975])
        return float(delta.mean()), ci, bool(ci[1] < 0 or ci[0] > 0), len(weekly)


    print('\n' + '=' * 70)
    print('STAGE 4 (SUPERSEDED — CONFIRMED SIGN-INVERTED, KEPT AS DIAGNOSTIC MILESTONE)')
    print('Higher surprise multiplier => smaller current mean correction, at the same prior state.')
    print('Exactly one observation update; forecast precedes this match result.')
    selected_sensitivity, stage4_validation, stage4_validation_folds, stage4_validation_predictions = (
        select_stage4_sensitivity(train_df, change_validation_folds))
    print('\nTRAINING-ONLY EXPANDING VALIDATION (weighted by held-out fold size)')
    print(stage4_validation.to_string(index=False, float_format=lambda x: f'{x:.7f}'))
    print(f'Selected sensitivity (log-loss only): {selected_sensitivity:g}')
    stage4_validation_delta, stage4_validation_ci, stage4_excludes_zero, stage4_validation_weeks = (
        stage4_logloss_interval(stage4_validation_predictions[selected_sensitivity],
                               stage4_validation_predictions[0.0]))
    print(f'Validation log-loss difference (selected minus zero): {stage4_validation_delta:.7f}')
    print(f'95% paired week-block bootstrap CI: [{stage4_validation_ci[0]:.7f}, {stage4_validation_ci[1]:.7f}]')
    print(f'Interval excludes zero: {stage4_excludes_zero}; weeks={stage4_validation_weeks}; resamples=2000')
    print('Selection-data interval: exploratory, not corrected for choosing the best sensitivity.')
    print('The interval is reported honestly; it does not gate selection.')

    stage4_predictions, stage4_final_team_states = run_sequential_filter_with_surprise(df, selected_sensitivity)
    stage4_test = stage4_predictions.loc[test_df.index]
    stage4_common_test = stage4_test.loc[stage2_common_test.index]
    pd.testing.assert_frame_equal(stage4_common_test[match_columns], stage2_common_test[match_columns])
    stage4_common_probs = stage4_common_test[prob_columns].to_numpy(float)
    stage4_metrics = probability_metrics(stage4_common_probs, common_actual_class)
    stage4_comparison = pd.concat([comparison, pd.DataFrame([dict(Model='Stage 4', **stage4_metrics)])], ignore_index=True)
    print(f'\nIDENTICAL-MATCH COMPARISON ({len(stage4_common_test)} matches)')
    print(stage4_comparison.to_string(index=False, float_format=lambda x: f'{x:.7f}'))
    print('Brier is mean squared error across all three classes; lower loss scores are better.')
    print('This test period has been inspected previously; results are retrospective, not a fresh holdout.')
    if selected_sensitivity == 0.0:
        pd.testing.assert_frame_equal(stage4_predictions[stage2_predictions.columns], stage2_predictions, check_exact=True)
        for team, state in final_team_states.items():
            if vars(stage4_final_team_states[team]) != vars(state):
                raise AssertionError('Zero sensitivity must recover every Stage 2 state exactly.')
        print('Zero-sensitivity recovery: all forecasts and final states exactly equal Stage 2.')

    # Reuse the exact Stage 3 diagnostic subset (383 of all 3829 held-out rows
    # in the frozen data), ranked solely by Stage 2's realized surprise.
    print(f'\nTop 10% most surprising matches (N={len(top_positions)}):')
    print(f'  Same existing Stage 2-ranked subset of all {len(stage2_test)} test matches.')
    print(f'  Stage 2 RPS on these: {stage2_rps(stage2_test.iloc[top_positions]):.7f}')
    print(f'  Stage 4 RPS on these: {stage2_rps(stage4_test.iloc[top_positions]):.7f}')

    # Primary material-adjustment comparison stays on the common test population.
    stage4_material_mask = (stage4_common_test.surprise_mult - 1.0).abs() > 0.05
    stage4_material_ids = stage4_common_test.index[stage4_material_mask]
    print(f'\nMatches with material surprise adjustment (N={len(stage4_material_ids)}):')
    print(f'  Population: the same {len(stage4_common_test)} common test matches.')
    stage4_material_metrics = {}
    for name, forecasts in [('Stage 2', stage2_common_test), ('Stage 4', stage4_common_test)]:
        if len(stage4_material_ids):
            subset = forecasts.loc[stage4_material_ids]
            values = probability_metrics(subset[prob_columns], outcome_classes(subset))
            stage4_material_metrics[name] = values
            print(f"  {name} RPS/Brier/LogLoss on these matches: "
                  f"{values['RPS']:.7f} / {values['Brier']:.7f} / {values['LogLoss']:.7f}")
        else:
            print(f'  {name} RPS/Brier/LogLoss on these matches: N/A (empty subset)')
    print('Both subsets depend on realized outcomes; these are descriptive, not causal effect estimates.')
    print('A match update cannot improve that same match forecast; only later forecasts can change.')
    print(f'Stage 4 test multiplier mean/min/max: {stage4_test.surprise_mult.mean():.7f} / '
          f'{stage4_test.surprise_mult.min():.7f} / {stage4_test.surprise_mult.max():.7f}')
    print('=' * 70)

    # STAGE 5: CURRENT-MATCH SURPRISE SCALES ONLY THE MEAN STEP
    # Reuse Stage 4's multiplier, candidates, and paired-bootstrap helper unchanged.


    def update_step_with_surprise_v2(home_state, away_state, goals_home, goals_away,
                                     lambda_home, mu_away, surprise_mult=1.0):
        if not np.isfinite(surprise_mult) or not 0.5 <= surprise_mult <= 2.0:
            raise ValueError('surprise_mult must be finite and in [0.5, 2.0].')
        home_gradient = goals_home - lambda_home
        away_gradient = goals_away - mu_away
        # Ordinary precision/variance update: surprise appears ONLY in mean steps.
        attack_home_precision = 1.0 / home_state.attack_var + lambda_home
        defence_away_precision = 1.0 / away_state.defence_var + lambda_home
        attack_away_precision = 1.0 / away_state.attack_var + mu_away
        defence_home_precision = 1.0 / home_state.defence_var + mu_away

        home_state.attack_mean += surprise_mult * home_gradient / attack_home_precision
        home_state.attack_var = 1.0 / attack_home_precision
        away_state.defence_mean += surprise_mult * home_gradient / defence_away_precision
        away_state.defence_var = 1.0 / defence_away_precision
        away_state.attack_mean += surprise_mult * away_gradient / attack_away_precision
        away_state.attack_var = 1.0 / attack_away_precision
        home_state.defence_mean += surprise_mult * away_gradient / defence_home_precision
        home_state.defence_var = 1.0 / defence_home_precision
        return home_state, away_state


    def run_sequential_filter_with_surprise_v2(df, sensitivity, *, home_adv_override=None):
        # Match Stage 4's optional validation-only override without mutating home_adv.
        surprise_to_update_mult(1.0, sensitivity)
        advantage = home_adv if home_adv_override is None else float(home_adv_override)
        if not np.isfinite(advantage):
            raise ValueError('home advantage must be finite.')
        if not df.index.is_unique:
            raise ValueError('Unique match indices are required for exact test alignment.')
        required = ['date', 'team_home', 'team_away', 'goals_home', 'goals_away']
        if df[required].isna().any().any():
            raise ValueError('Stage 5 requires complete dates, teams, and goals.')
        goals_array = df[['goals_home', 'goals_away']].to_numpy(dtype=float)
        if (not np.isfinite(goals_array).all() or (goals_array < 0).any()
                or (goals_array != np.floor(goals_array)).any()):
            raise ValueError('Poisson observations must be finite nonnegative integer goals.')
        if (df.team_home == df.team_away).any():
            raise ValueError('A match must contain two different teams.')
        ordered_df = df.sort_values('date', kind='stable')
        team_states, records = {}, []
        for row in ordered_df.itertuples():
            if row.team_home not in team_states:
                team_states[row.team_home] = TeamState()
            if row.team_away not in team_states:
                team_states[row.team_away] = TeamState()
            home_state = predict_step(team_states[row.team_home], PROCESS_NOISE)
            away_state = predict_step(team_states[row.team_away], PROCESS_NOISE)
            lambda_home = np.exp(home_state.attack_mean + away_state.defence_mean + advantage)
            mu_away = np.exp(away_state.attack_mean + home_state.defence_mean)
            try:
                pred_h, pred_d, pred_a = match_prediction_probs(home_state, away_state, advantage)
            except FloatingPointError:
                # Identical normalized 0..10 score grid, evaluated in log space only
                # when ordinary PMFs underflow. Do NOT cap rates or change updates.
                if not np.isfinite([lambda_home, mu_away]).all():
                    raise
                goals = np.arange(11)
                log_grid = (poisson.logpmf(goals, lambda_home)[:, None]
                            + poisson.logpmf(goals, mu_away)[None, :])
                grid = np.exp(log_grid - np.max(log_grid))
                mass = grid.sum()
                if not np.isfinite(mass) or mass <= 0:
                    raise FloatingPointError('Stage 5 log-space score grid is invalid.')
                grid /= mass
                pred_h, pred_d, pred_a = float(np.tril(grid, -1).sum()), float(np.trace(grid)), float(np.triu(grid, 1).sum())
                print(f'Stage 5 numerical fallback (not a rate cap): sensitivity={sensitivity:g}, '
                      f'{row.date}, {row.team_home} vs {row.team_away}, '
                      f'goal rates={lambda_home:.7g}/{mu_away:.7g}', flush=True)
            records.append(dict(date=row.date, team_home=row.team_home, team_away=row.team_away,
                                goals_home=row.goals_home, goals_away=row.goals_away,
                                pred_home_win=pred_h, pred_draw=pred_d, pred_away_win=pred_a))
            # Only after freezing the forecast may the result affect surprise/state.
            raw_surprise = prediction_surprise_score(pred_h, pred_d, pred_a, row.goals_home, row.goals_away)
            mult = surprise_to_update_mult(raw_surprise, sensitivity)
            records[-1]['surprise_mult'] = mult
            home_state, away_state = update_step_with_surprise_v2(
                home_state, away_state, row.goals_home, row.goals_away, lambda_home, mu_away, mult)
            team_states[row.team_home], team_states[row.team_away] = home_state, away_state
        columns = required + ['pred_home_win', 'pred_draw', 'pred_away_win', 'surprise_mult']
        return pd.DataFrame(records, index=ordered_df.index, columns=columns), team_states


    def select_stage5_sensitivity(training_rows, existing_fold_rows):
        # Same fold plan and log-loss-only protocol; no changes to Stage 4's selector.
        plan = existing_fold_rows[['fold', 'start', 'end', 'home_adv']].drop_duplicates().sort_values('fold')
        if len(plan) != 3 or not plan.fold.is_unique:
            raise ValueError('Expected the three unique existing expanding folds.')
        fold_results = []
        held_out = {s: [] for s in SENSITIVITY_CANDIDATES}
        for fold in plan.itertuples(index=False):
            start, end = int(fold.start), int(fold.end)
            if not 0 < start < end <= len(training_rows):
                raise ValueError('Validation fold leaves the training dataset.')
            prefix = training_rows.iloc[:start]
            expected_adv = float(np.log(prefix.goals_home.mean()) - np.log(prefix.goals_away.mean()))
            if not np.isclose(expected_adv, fold.home_adv, rtol=0, atol=1e-14):
                raise ValueError('Validation home advantage must use only its training prefix.')
            actual = training_rows.iloc[start:end]
            for sensitivity in SENSITIVITY_CANDIDATES:
                forecasts, _ = run_sequential_filter_with_surprise_v2(
                    training_rows.iloc[:end], sensitivity, home_adv_override=fold.home_adv)
                held = forecasts.loc[actual.index]
                values = probability_metrics(held[prob_columns], outcome_classes(actual))
                held_out[sensitivity].append(held)
                fold_results.append(dict(fold=int(fold.fold), sensitivity=sensitivity, n=len(actual),
                                         start=start, end=end, home_adv=float(fold.home_adv), **values))
            print(f'Stage 5 validation fold {fold.fold}: prefix={start}, held-out={end-start} complete.', flush=True)
        folds = pd.DataFrame(fold_results)
        summaries = []
        for sensitivity in SENSITIVITY_CANDIDATES:
            rows = folds.loc[folds.sensitivity == sensitivity]
            summaries.append(dict(sensitivity=sensitivity, **{
                key: float(np.average(rows[key], weights=rows.n))
                for key in ('RPS', 'Brier', 'LogLoss', 'Accuracy')}))
        summary = pd.DataFrame(summaries)
        selected = float(summary.loc[summary.LogLoss.idxmin(), 'sensitivity'])
        pooled = {s: pd.concat(parts) for s, parts in held_out.items()}
        if any(not rows.index.is_unique for rows in pooled.values()):
            raise ValueError('Validation observations must not be counted twice.')
        return selected, summary, folds, pooled


    print('\n' + '=' * 70)
    print('STAGE 5: CORRECTED DIRECTION — SURPRISE SCALES THE MEAN STEP')
    print('Higher multiplier => larger signed mean step, at the same prior state and goal rates.')
    print('Precision and variance use the ordinary unscaled Fisher information.')
    print('Exactly one observation update; forecast precedes this match result.')
    stage5_selected_sensitivity, stage5_validation, stage5_validation_folds, stage5_validation_predictions = (
        select_stage5_sensitivity(train_df, change_validation_folds))
    print('\nTRAINING-ONLY EXPANDING VALIDATION (weighted by held-out fold size)')
    print(stage5_validation.to_string(index=False, float_format=lambda x: f'{x:.7f}'))
    print(f'Stage 5 selected sensitivity (log-loss only): {stage5_selected_sensitivity:g}')
    stage5_validation_delta, stage5_validation_ci, stage5_excludes_zero, stage5_validation_weeks = (
        stage4_logloss_interval(stage5_validation_predictions[stage5_selected_sensitivity],
                               stage5_validation_predictions[0.0]))
    print(f'Validation log-loss difference (selected minus zero): {stage5_validation_delta:.7f}')
    print(f'95% paired week-block bootstrap CI: [{stage5_validation_ci[0]:.7f}, {stage5_validation_ci[1]:.7f}]')
    print(f'Interval excludes zero: {stage5_excludes_zero}; weeks={stage5_validation_weeks}; resamples=2000')
    print('Selection-data interval: exploratory, not corrected for choosing the best sensitivity.')
    print('The interval is reported honestly; it does not gate selection.')

    stage5_predictions, stage5_final_team_states = run_sequential_filter_with_surprise_v2(df, stage5_selected_sensitivity)
    stage5_test = stage5_predictions.loc[test_df.index]
    stage5_common_test = stage5_test.loc[stage2_common_test.index]
    pd.testing.assert_frame_equal(stage5_common_test[match_columns], stage2_common_test[match_columns])
    stage5_common_probs = stage5_common_test[prob_columns].to_numpy(float)
    stage5_metrics = probability_metrics(stage5_common_probs, common_actual_class)
    stage5_comparison = pd.concat([stage4_comparison, pd.DataFrame([dict(Model='Stage 5', **stage5_metrics)])], ignore_index=True)
    print(f'\nIDENTICAL-MATCH COMPARISON ({len(stage5_common_test)} matches)')
    print(stage5_comparison.to_string(index=False, float_format=lambda x: f'{x:.7f}'))
    print('Brier is mean squared error across all three classes; lower loss scores are better.')
    print('This test period has been inspected previously; results are retrospective, not a fresh holdout.')
    if stage5_selected_sensitivity == 0.0:
        pd.testing.assert_frame_equal(stage5_predictions[stage2_predictions.columns], stage2_predictions, check_exact=True)
        for team, state in final_team_states.items():
            if vars(stage5_final_team_states[team]) != vars(state):
                raise AssertionError('Zero sensitivity must recover every Stage 2 state exactly.')
        print('Zero selected: surprise scaling is inactive; all forecasts and final states exactly equal Stage 2.')

    print(f'\nTop 10% most surprising matches (N={len(top_positions)}):')
    print(f'  Same existing Stage 2-ranked subset of all {len(stage2_test)} test matches.')
    print(f'  Stage 2 RPS on these: {stage2_rps(stage2_test.iloc[top_positions]):.7f}')
    print(f'  Stage 5 RPS on these: {stage2_rps(stage5_test.iloc[top_positions]):.7f}')

    stage5_material_mask = (stage5_common_test.surprise_mult - 1.0).abs() > 0.05
    stage5_material_ids = stage5_common_test.index[stage5_material_mask]
    print(f'\nMatches with material surprise adjustment (N={len(stage5_material_ids)}):')
    print(f'  Population: the same {len(stage5_common_test)} common test matches.')
    stage5_material_metrics = {}
    for name, forecasts in [('Stage 2', stage2_common_test), ('Stage 5', stage5_common_test)]:
        if len(stage5_material_ids):
            subset = forecasts.loc[stage5_material_ids]
            values = probability_metrics(subset[prob_columns], outcome_classes(subset))
            stage5_material_metrics[name] = values
            print(f"  {name} RPS/Brier/LogLoss on these matches: "
                  f"{values['RPS']:.7f} / {values['Brier']:.7f} / {values['LogLoss']:.7f}")
        else:
            print(f'  {name} RPS/Brier/LogLoss on these matches: N/A (empty subset)')
    print('Both subsets depend on realized outcomes; these are descriptive, not causal effect estimates.')
    print('Correct update direction does not guarantee better forecasts on this outcome-selected subset.')
    print('A match update cannot improve that same match forecast; only later forecasts can change.')
    stage5_theoretical_min = surprise_to_update_mult(0.2, stage5_selected_sensitivity)
    stage5_theoretical_max = surprise_to_update_mult(5.0, stage5_selected_sensitivity)
    if not stage5_test.surprise_mult.between(stage5_theoretical_min, stage5_theoretical_max).all():
        raise AssertionError('Stage 5 multipliers escaped their sensitivity-specific bounds.')
    print(f'Stage 5 test multiplier mean/min/max: {stage5_test.surprise_mult.mean():.7f} / '
          f'{stage5_test.surprise_mult.min():.7f} / {stage5_test.surprise_mult.max():.7f}')
    print(f'Theoretical range at sensitivity {stage5_selected_sensitivity:g}: '
          f'[{stage5_theoretical_min:.7f}, {stage5_theoretical_max:.7f}]')
    print('=' * 70)

if __name__ == "__main__":
    main()

Execution data: 56 SHA-256-verified cached upstream fixture snapshots; no fresh network fetch.
PROJECT B12: EXTRACTING 2019-2026 DATASET METRICS

[1/4] Scraping/Ingesting match data across 7 seasons...
  ✓ Ingested ENG Premier League 2019-2020
  ✓ Ingested ENG Premier League 2020-2021
  ✓ Ingested ENG Premier League 2021-2022
  ✓ Ingested ENG Premier League 2022-2023
  ✓ Ingested ENG Premier League 2023-2024
  ✓ Ingested ENG Premier League 2024-2025
  ✓ Ingested ENG Premier League 2025-2026
  ✓ Ingested ESP La Liga 2019-2020
  ✓ Ingested ESP La Liga 2020-2021
  ✓ Ingested ESP La Liga 2021-2022
  ✓ Ingested ESP La Liga 2022-2023
  ✓ Ingested ESP La Liga 2023-2024
  ✓ Ingested ESP La Liga 2024-2025
  ✓ Ingested ESP La Liga 2025-2026
  ✓ Ingested DEU Bundesliga 1 2019-2020
  ✓ Ingested DEU Bundesliga 1 2020-2021
  ✓ Ingested DEU Bundesliga 1 2021-2022
  ✓ Ingested DEU Bundesliga 1 2022-2023
  ✓ Ingested DEU Bundesliga 1 2023-2024
  ✓ Ingested DEU Bundesliga 1 2024-2025
  ✓ Ingested DEU Bu

# Single-update change-detector verification — 2026-09-22

## Implemented changes

- Removed the extra gradient/Fisher surprise-weighting code from active Stage 3.
- Exactly one call to the original observation update per match.
- Persistent signed residuals and repeated surprise can increase future process variance.
- Bounded variance boosts decay when persistent evidence disappears.
- Optional timestamp-checked pre-match lineup offsets; historical efficacy remains unevaluated.
- Four-way ablation scaffold, per-class calibration, draw-specific scores, and closing-market comparison.
- Paired calendar-week confidence intervals and actual subsequent-match upset diagnostics.

The Poisson residual remains signed and unscaled. The detector records evidence after a match and only changes later priors. It is not a second observation update. Stage 1/2 source is unchanged.

## Training-only validation

Three expanding folds within the 15,316 training matches evaluate 6,893 later training matches. Home advantage in each fold uses only that fold's earlier prefix. The selection rule remains strict: no aggregate deterioration in RPS, Brier, log-loss or argmax accuracy. RPS breaks ties among eligible candidates.

| Scale | RPS | Brier | LogLoss | Accuracy | Eligible |
|---:|---:|---:|---:|---:|:---:|
| 0 | 0.2023143 | 0.1969495 | 0.9979680 | 52.3865% | True |
| 0.25 | 0.2023124 | 0.1969482 | 0.9979532 | 52.3720% | False |
| 0.5 | 0.2023116 | 0.1969476 | 0.9979441 | 52.3575% | False |
| 1 | 0.2023121 | 0.1969482 | 0.9979366 | 52.3575% | False |
| 2 | 0.2023191 | 0.1969535 | 0.9979467 | 52.3284% | False |

Selected CHANGE_SCALE = **0**.

The nonzero scales 0.25, 0.5 and 1 produced tiny improvements in all three validation probability scores but lost one or two correct outcome labels. They failed the existing no-degradation guard. This is not proof that the detector is ineffective; nor is it evidence of a reliable predictive gain. The selected deployment still equals Stage 2.

## Identical 3,545-match retrospective comparison

| Model | RPS | Brier | LogLoss | Accuracy |
|---|---:|---:|---:|---:|
| Stage 1 | 0.2057245 | 0.1994488 | 1.0022933 | 50.9732% |
| Stage 2 | 0.2060203 | 0.1998337 | 1.0056991 | 51.8759% |
| Stage 3 | 0.2060203 | 0.1998337 | 1.0056991 | 51.8759% |

Test matches receiving extra variance: 0/3829.
Paired week-block Stage 3 minus Stage 2 RPS interval: [0.0, 0.0].

## Independent checks

- 21 regression/property tests passed.
- A separate matrix implementation and normalized Poisson calculation reproduced all 19,145 forecasts.
- Every validation fold, selection result, raw surprise and applied extra variance was independently checked.
- Ordinary update call count equals match count; forbidden amplification code is absent.
- Single shocks do not trigger extra uncertainty from a neutral detector state.
- Persistent same-direction residuals trigger bounded boosts; boosts decay without continuing evidence.
- Score changes cannot affect the forecast of that same match or earlier matches.
- Disconnected teams remain unaffected.
- Lineup effects apply once, residuals use adjusted expected goals, and effects are not stored as permanent team offsets.
- Late, duplicate, unmatched, missing, oversized and NaT lineup inputs are rejected.
- Metrics, confusion matrices, calibration totals and market-odds normalization were independently verified.

An additional timestamp-hardening check was applied after the real-data execution and verified with targeted tests. No lineup data was used in the real-data run; its model path and reported scores are unchanged by that input-validation check.

## Controlled stress paths

Nine paths test isolated shocks, repeated underperformance, and alternating score patterns at variance scales 0, 1 and 2. These synthetic diagnostics test causality and stability; they do not establish real predictive benefit. Alternating observed scores need not yield alternating signed residuals after team strengths adapt.

| Scenario | Scale | Boosted matches | Max probability shift | Late shift |
|---|---:|---:|---:|---:|
| isolated | 0 | 0 | 0.540477 | 0.003607 |
| isolated | 1 | 0 | 0.540477 | 0.003607 |
| isolated | 2 | 0 | 0.540477 | 0.003607 |
| persistent | 0 | 0 | 0.801495 | 0.036690 |
| persistent | 1 | 38 | 0.801715 | 0.029594 |
| persistent | 2 | 38 | 0.801919 | 0.027339 |
| alternating | 0 | 0 | 0.585952 | 0.018040 |
| alternating | 1 | 27 | 0.585952 | 0.017210 |
| alternating | 2 | 27 | 0.585952 | 0.016428 |

## Closing market and draw calibration

The closing-market comparison uses normalized inverse average odds on identical eligible rows. These prices can contain later information than a T-60 forecast; they never enter training or parameter selection. C-column semantics are documented by [Football-Data](https://www.football-data.co.uk/data).

| Model | RPS | Brier | LogLoss | Accuracy |
|---|---:|---:|---:|---:|
| Market closing average (normalized) | 0.1926041 | 0.1898924 | 0.9598217 | 54.7814% |
| Stage 2 on identical market rows | 0.2060203 | 0.1998337 | 1.0056991 | 51.8759% |
| Stage 3 on identical market rows | 0.2060203 | 0.1998337 | 1.0056991 | 51.8759% |

Stage 2/selected Stage 3 average draw probability is 22.76%, versus 25.05% actual draw frequency on the common test set. This supplies evidence of average draw underprediction; low draw argmax recall alone would not establish that. Full reliability bins and draw binary losses are included in the notebook output.

## Remaining dependencies

- Historical lineup/player data is not available. The integration is prepared and tested, but player-effect estimation and real lineup ablations cannot be claimed complete.
- A fresh, untouched prospective evaluation period remains necessary. The current test period has been reviewed repeatedly and is labeled retrospective.
- No nonzero detector setting met all existing acceptance conditions. No positive novelty claim is justified by this run.

See LINEUP_INTEGRATION_AND_EVALUATION.md for the input schema, clock convention, availability cutoff, rating-estimation requirements and prospective evaluation protocol.


# Lineup integration and evaluation contract

## What is implemented

Stage 3 performs exactly one ordinary Poisson observation update per match.
No surprise multiplier is applied to its gradient or Fisher information.
A separate persistent-change detector can increase process variance at later
appearances, based solely on previous matches. It requires at least five
observations, three consecutive material residuals in the same direction,
at least two surprises in the last five appearances, and a nontrivial EWMA.
Residuals are clipped to [-3, 3] for the detector only. The actual observation
gradient remains unmodified. A triggered boost decays by half per appearance
without persistent evidence, and extra process variance is bounded.

Validation chooses a variance scale from 0, 0.25, 0.5, 1, 2. The base process
variance is 0.01; the largest possible total addition per component per
appearance is therefore 0.03. Scale zero recovers Stage 2 exactly. It is an
allowed outcome, not a claimed predictive gain.

## Optional lineup input

No historical lineup dataset is present. The implemented interface consumes
precomputed lineup effects; it does not invent player ratings or train player
effects without observations. Supply a pandas DataFrame named
`LINEUP_ADJUSTMENTS` in a Colab cell before running the model cell.

| Column | Meaning |
|---|---|
| date | Match kickoff, exactly matching the fixture clock |
| competition | Exact competition string in the match data |
| team_home, team_away | Exact fixture team names |
| available_at | When this lineup snapshot became available |
| ratings_asof | Latest information timestamp used to estimate the effects |
| home_attack_delta | Home attack log-rate difference from its usual lineup |
| home_defence_delta | Home concession-tendency log-rate difference |
| away_attack_delta | Away attack log-rate difference |
| away_defence_delta | Away concession-tendency log-rate difference |

All four effects must be finite and between -0.5 and +0.5. These are log-rate
effects, not raw player scores. A +0.1 attack effect multiplies the associated
goal rate by exp(0.1). A positive defence effect means more goals conceded.
Home log-rate receives home attack plus away defence; away log-rate receives
away attack plus home defence. Offsets apply once to the match prediction and
to the expected rates used by the residual. They are not added permanently
to team means.

Both timestamps must be at or before kickoff minus 60 minutes. If confirmed
lineups arrive later, they are not available for this forecast cutoff. Select
one eligible snapshot per fixture. Duplicate or unmatched fixtures, invalid
timestamps (including strings that parse to NaT), oversized effects and future
information are rejected. Missing fixtures use zero offsets and are reported
as missing coverage. Zero offset is a fallback, not evidence of full strength.

Normalize time zones consistently: the interface interprets naive timestamps
as UTC, but it cannot infer or repair an upstream fixture clock. Verify the
source clock before merging live lineup records. Timestamp checks also cannot
prove that an external rating was trained without future information; the
rating-estimation pipeline must independently enforce that constraint.

Example for a real, prepared file (no example records are fabricated):

```python
import pandas as pd
LINEUP_ADJUSTMENTS = pd.read_csv('/content/lineup_adjustments.csv')
```

## Remaining player-data work

Obtain historical starting XIs, stable player IDs, roles, minutes and reliable
availability timestamps. Estimate player/replacement contributions only on
earlier matches, using regularization and a fallback for unseen players.
Compare the actual XI to an earlier-data usual-lineup reference to avoid
double-counting strength already captured in the team state. No player effect
estimator is currently claimed to exist.

## Honest model comparisons

The notebook prints four ablations: baseline, baseline plus detector,
baseline plus lineup, and both together. Without lineup data, the last two
are explicitly NOT EVALUATED. With partial data it reports coverage. The
detector scale is frozen from no-lineup training validation; a later joint
validation study is required before claiming the best combined configuration.

It also reports RPS, three-class mean Brier, log-loss, argmax classification,
confusion matrices, draw-specific scores, reliability bins, and paired
calendar-week bootstrap intervals. Post-upset diagnostics evaluate subsequent
team appearances. The top-surprise subgroup is labeled retrospective and
must not be used as proof that adaptation caused better predictions.

Closing-market average odds are normalized by dividing inverse odds by their
sum, on exactly the rows available to all compared models. This is a simple
margin-removal approximation. Closing prices may contain information that was
unavailable 60 minutes before kickoff; this is a later-information benchmark,
not a same-cutoff trading comparison. They never enter model training or
parameter selection. Football-Data documents C-column closing prices here:
https://www.football-data.co.uk/data

## Fresh evidence still needed

The current test dates have been examined in previous iterations. All new
parameter selection remains inside the chronological training prefix, but
the existing test period is a retrospective benchmark, not a fresh holdout.
Before a prospective claim, freeze the code, ratings procedure, forecast
cutoff, primary metric, and planned evaluation interval. Save forecasts before
kickoff, use only results available at forecast time, and score a new period
without tuning against its outcomes. External unseen competition data is
another option, with league-transfer limitations stated explicitly.

No synthetic lineup integration test establishes real predictive benefit.
No stability test guarantees that every subsequent prediction improves.
